# Sail Comparison: PyPI (normal parquet) vs Local build (bucketed parquet)

Compare write + query performance between:
- **Baseline (PyPI Sail)**: normal parquet write + TPC-H Q5
- **Local build**: bucketed parquet write (CLUSTERED BY) + TPC-H Q5

Bucketed tables: `orders` (o_orderkey) and `lineitem` (l_orderkey) with 64 buckets.

## Prerequisites

Build the local Sail wheel (from the Sail repo):
```bash
hatch run maturin build --release
```

Update `SAIL_LOCAL_PATH` in cell 1 to point to your Sail repo.

## Workflow
1. Run cells 1-3 (generate data + baseline with PyPI Sail)
2. Run cell 4 (install local build) — kernel restarts automatically
3. Run cells 5-6 (re-setup + local build bucketed run)
4. Run cell 7 (comparison)
5. Run cell 8 (restore PyPI Sail)

In [4]:
import os
import time
import json

BASE = os.getcwd()
DATA_DIR = os.path.join(BASE, "local_data")
WORKING_DIR = os.path.join(BASE, "local_working_dir")
RESULTS_FILE = os.path.join(WORKING_DIR, "sail_results.json")
SCALE_FACTORS = [10, 50]
NUM_BUCKETS = 64
SAIL_LOCAL_PATH = "/Users/davidlopez/Proyectos/sail"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(WORKING_DIR, exist_ok=True)

# TPC-H tables: (name, bucket_key or None)
TPCH_TABLES = [
    ("customer", None),
    ("lineitem", "l_orderkey"),
    ("nation", None),
    ("orders", "o_orderkey"),
    ("part", None),
    ("partsupp", None),
    ("region", None),
    ("supplier", None),
]

TPCH_Q5 = """
SELECT
    n_name,
    SUM(l_extendedprice * (1 - l_discount)) AS revenue
FROM
    customer
    INNER JOIN orders ON c_custkey = o_custkey
    INNER JOIN lineitem ON l_orderkey = o_orderkey
    INNER JOIN supplier ON l_suppkey = s_suppkey
    INNER JOIN nation ON c_nationkey = s_nationkey AND s_nationkey = n_nationkey
    INNER JOIN region ON n_regionkey = r_regionkey
WHERE
    r_name = 'ASIA'
    AND o_orderdate >= CAST('1994-01-01' AS DATE)
    AND o_orderdate < CAST('1995-01-01' AS DATE)
GROUP BY
    n_name
ORDER BY
    revenue DESC
"""

## 1. Generate TPC-H data (run once)

In [5]:
from lakebench.datagen import TPCHDataGenerator

for sf in SCALE_FACTORS:
    target = os.path.join(DATA_DIR, f"tpch_sf{sf}")
    if not os.path.exists(target):
        print(f"Generating TPC-H SF={sf}...")
        TPCHDataGenerator(scale_factor=sf, target_folder_uri=target).run()
    else:
        print(f"TPC-H SF={sf} already exists, skipping")

Generating TPC-H SF=10...
🚀 Starting parallel generation of 8 tables with multithreading...
📊 Scale Factor: 10
📁 Output Directory: /Users/davidlopez/Proyectos/LakeBench/examples/local_data/tpch_sf10
🔧 customer - Using 1 parts (target file size: 128mb)
🔧 lineitem - Using 12 parts (target file size: 128mb)
🔧 nation - Using 1 parts (target file size: 128mb)
🔧 orders - Using 3 parts (target file size: 128mb)
🔧 part - Using 1 parts (target file size: 128mb)
🔧 partsupp - Using 2 parts (target file size: 128mb)
🔧 region - Using 1 parts (target file size: 128mb)
🔧 supplier - Using 1 parts (target file size: 128mb)
✅ nation - Generation completed successfully
✅ region - Generation completed successfully
✅ supplier - Generation completed successfully
✅ customer - Generation completed successfully
✅ part - Generation completed successfully
✅ partsupp - Generation completed successfully
✅ orders - Generation completed successfully
✅ lineitem - Generation completed successfully

📋 Generation Summar

## 2. Baseline: PyPI Sail (normal parquet write + Q5)

Starts the Sail server, writes all tables as normal parquet, runs Q5, stops server, and saves results to disk.

In [6]:
from pysail.spark import SparkConnectServer
from pyspark.sql import SparkSession
import pysail

print(f"Sail version: {pysail.__version__}")

server = SparkConnectServer(port=50051)
server.start(background=True)
host, port = server.listening_address
spark = SparkSession.builder.remote(f"sc://{host}:{port}").getOrCreate()

pypi_write = []
pypi_query = []

for sf in SCALE_FACTORS:
    print(f"\n{'='*60}")
    print(f"Baseline (PyPI) normal write SF={sf}")
    print(f"{'='*60}")
    input_dir = os.path.join(DATA_DIR, f"tpch_sf{sf}")
    output_dir = os.path.join(WORKING_DIR, f"sail_pypi_sf{sf}")
    os.makedirs(output_dir, exist_ok=True)

    for table_name, _ in TPCH_TABLES:
        src = os.path.join(input_dir, table_name)
        dst = os.path.join(output_dir, table_name)
        t0 = time.perf_counter()
        spark.read.parquet(src).write.format("parquet").mode("overwrite").save(dst)
        ms = int((time.perf_counter() - t0) * 1000)
        pypi_write.append({"sf": sf, "table": table_name, "ms": ms})
        print(f"  {table_name}: {ms}ms")

    for table_name, _ in TPCH_TABLES:
        dst = os.path.join(output_dir, table_name)
        spark.read.parquet(dst).createOrReplaceTempView(table_name)

    print(f"\n  Running Q5...")
    spark.sql(TPCH_Q5).toPandas()  # warmup
    for i in range(3):
        t0 = time.perf_counter()
        spark.sql(TPCH_Q5).toPandas()
        ms = int((time.perf_counter() - t0) * 1000)
        pypi_query.append({"sf": sf, "run": i + 1, "ms": ms})
        print(f"  Q5 run {i+1}: {ms}ms")

server.stop()
print("\nServer stopped")

# Save baseline results to disk (survives kernel restart)
results = {"pypi_version": pysail.__version__, "pypi_write": pypi_write, "pypi_query": pypi_query}
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)
print(f"Baseline results saved to {RESULTS_FILE}")

Sail version: 0.5.1


[2026-02-28T20:09:24Z INFO sail_python::spark::server] Starting the Spark Connect server on 127.0.0.1:50051...



Baseline (PyPI) normal write SF=10


[2026-02-28T20:09:25Z INFO sail_session::session_manager::actor::handler] creating session 0bcba230-129a-4370-b988-ef0cae742217


  customer: 434ms
  lineitem: 4682ms
  nation: 2ms
  orders: 1091ms
  part: 263ms
  partsupp: 757ms
  region: 2ms
  supplier: 36ms

  Running Q5...
  Q5 run 1: 567ms
  Q5 run 2: 571ms
  Q5 run 3: 548ms

Baseline (PyPI) normal write SF=50
  customer: 812ms
  lineitem: 22339ms
  nation: 5ms
  orders: 5730ms
  part: 703ms
  partsupp: 4360ms
  region: 4ms
  supplier: 105ms

  Running Q5...
  Q5 run 1: 2899ms
  Q5 run 2: 2829ms
  Q5 run 3: 2815ms


[2026-02-28T20:10:20Z INFO sail_python::spark::server] Shutting down the Spark Connect server...



Server stopped
Baseline results saved to /Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_results.json


[2026-02-28T20:10:25Z INFO sail_python::spark::server] The Spark Connect server has stopped.


## 3. Install local Sail build

Installs the local wheel and restarts the kernel automatically. Continue from cell 5 after restart.

In [7]:
import subprocess
import glob
import os

SAIL_LOCAL_PATH = "/Users/davidlopez/Proyectos/sail"

# Find latest wheel from local build
wheels = sorted(glob.glob(os.path.join(SAIL_LOCAL_PATH, "target/wheels/*.whl")), key=os.path.getmtime)
if wheels:
    wheel = wheels[-1]
    print(f"Installing wheel: {os.path.basename(wheel)}")
    subprocess.run(["uv", "pip", "install", "--force-reinstall", "--no-cache", wheel], check=True)
else:
    raise FileNotFoundError(f"No wheel found in {SAIL_LOCAL_PATH}/target/wheels/. Run 'hatch run maturin build --release' in the Sail repo first.")

print("Local Sail installed. Restarting kernel...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

Installing wheel: pysail-0.5.1-cp38-abi3-macosx_11_0_arm64.whl


Using Python 3.13.11 environment at: /Users/davidlopez/Proyectos/LakeBench/.venv
Resolved 1 package in 16ms


Local Sail installed. Restarting kernel...


Prepared 1 package in 263ms
Uninstalled 1 package in 42ms
Installed 1 package in 6ms
 ~ pysail==0.5.1 (from file:///Users/davidlopez/Proyectos/sail/target/wheels/pysail-0.5.1-cp38-abi3-macosx_11_0_arm64.whl)


{'status': 'ok', 'restart': True}

## 4. Re-setup after kernel restart

Re-defines all variables (lost after kernel restart).

In [1]:
import os
import time
import json

BASE = os.getcwd()
DATA_DIR = os.path.join(BASE, "local_data")
WORKING_DIR = os.path.join(BASE, "local_working_dir")
RESULTS_FILE = os.path.join(WORKING_DIR, "sail_results.json")
SCALE_FACTORS = [10, 50]
NUM_BUCKETS = 64

TPCH_TABLES = [
    ("customer", None),
    ("lineitem", "l_orderkey"),
    ("nation", None),
    ("orders", "o_orderkey"),
    ("part", None),
    ("partsupp", None),
    ("region", None),
    ("supplier", None),
]

TPCH_Q5 = """
SELECT
    n_name,
    SUM(l_extendedprice * (1 - l_discount)) AS revenue
FROM
    customer
    INNER JOIN orders ON c_custkey = o_custkey
    INNER JOIN lineitem ON l_orderkey = o_orderkey
    INNER JOIN supplier ON l_suppkey = s_suppkey
    INNER JOIN nation ON c_nationkey = s_nationkey AND s_nationkey = n_nationkey
    INNER JOIN region ON n_regionkey = r_regionkey
WHERE
    r_name = 'ASIA'
    AND o_orderdate >= CAST('1994-01-01' AS DATE)
    AND o_orderdate < CAST('1995-01-01' AS DATE)
GROUP BY
    n_name
ORDER BY
    revenue DESC
"""

print("Setup ready")

Setup ready


## 5. Local build: bucketed parquet write + Q5

Starts the Sail server (now with local build), writes bucketed parquet, runs Q5, stops server, and saves results.

In [2]:
from pysail.spark import SparkConnectServer
from pyspark.sql import SparkSession
import pysail

print(f"Sail version: {pysail.__version__}")

server = SparkConnectServer(port=50051)
server.start(background=True)
host, port = server.listening_address
spark = SparkSession.builder.remote(f"sc://{host}:{port}").getOrCreate()

local_write = []
local_query = []

for sf in SCALE_FACTORS:
    print(f"\n{'='*60}")
    print(f"Local build bucketed write SF={sf} ({NUM_BUCKETS} buckets)")
    print(f"{'='*60}")
    input_dir = os.path.join(DATA_DIR, f"tpch_sf{sf}")
    output_dir = os.path.join(WORKING_DIR, f"sail_local_sf{sf}")
    os.makedirs(output_dir, exist_ok=True)

    for table_name, bucket_key in TPCH_TABLES:
        src = os.path.join(input_dir, table_name)
        dst = os.path.join(output_dir, table_name)
        df = spark.read.parquet(src)

        t0 = time.perf_counter()
        if bucket_key:
            # Use unique catalog name per SF to avoid path conflicts
            catalog_name = f"{table_name}_sf{sf}"
            spark.sql(f"DROP TABLE IF EXISTS {catalog_name}")
            (df.write.format("parquet")
                .bucketBy(NUM_BUCKETS, bucket_key)
                .mode("overwrite")
                .option("path", dst)
                .saveAsTable(catalog_name))
        else:
            df.write.format("parquet").mode("overwrite").save(dst)
        ms = int((time.perf_counter() - t0) * 1000)
        local_write.append({"sf": sf, "table": table_name, "bucketed": bucket_key is not None, "ms": ms})
        tag = f" (bucketed by {bucket_key})" if bucket_key else ""
        print(f"  {table_name}{tag}: {ms}ms")

    # Register tables and run Q5
    for table_name, _ in TPCH_TABLES:
        dst = os.path.join(output_dir, table_name)
        spark.read.parquet(dst).createOrReplaceTempView(table_name)

    print(f"\n  Running Q5...")
    spark.sql(TPCH_Q5).toPandas()  # warmup
    for i in range(3):
        t0 = time.perf_counter()
        spark.sql(TPCH_Q5).toPandas()
        ms = int((time.perf_counter() - t0) * 1000)
        local_query.append({"sf": sf, "run": i + 1, "ms": ms})
        print(f"  Q5 run {i+1}: {ms}ms")

server.stop()
print("\nServer stopped")

# Load baseline and add local results
with open(RESULTS_FILE) as f:
    results = json.load(f)
results["local_version"] = pysail.__version__
results["local_write"] = local_write
results["local_query"] = local_query
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)
print(f"Local results saved to {RESULTS_FILE}")

Sail version: 0.5.1


[2026-02-28T20:10:46Z INFO sail_python::spark::server] Starting the Spark Connect server on 127.0.0.1:50051...



Local build bucketed write SF=10 (64 buckets)


[2026-02-28T20:10:46Z INFO sail_session::session_manager::actor::handler] creating session 010cae45-062a-4b09-b476-3c5162b50c91


  customer: 344ms


...

[2026-02-28T20:11:02Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 0: 458338 rows to Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_local_sf50/lineitem/bucket_00000.parquet
[2026-02-28T20:11:03Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 1: 461568 rows to Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_local_sf50/lineitem/bucket_00001.parquet
[2026-02-28T20:11:05Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 2: 461353 rows to Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_local_sf50/lineitem/bucket_00002.parquet
[2026-02-28T20:11:05Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 3: 459379 rows to Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_local_sf50/lineitem/bucket_00003.parquet
[2026-02-28T20:11:05Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 4: 458522 rows to Users/davidlope

  lineitem (bucketed by l_orderkey): 24626ms
  nation: 3ms


[2026-02-28T20:11:21Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 0: 156401 rows to Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_local_sf50/orders/bucket_00000.parquet
[2026-02-28T20:11:21Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 1: 156571 rows to Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_local_sf50/orders/bucket_00001.parquet
[2026-02-28T20:11:21Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 2: 156016 rows to Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_local_sf50/orders/bucket_00002.parquet
[2026-02-28T20:11:21Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 3: 155654 rows to Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_local_sf50/orders/bucket_00003.parquet
[2026-02-28T20:11:21Z INFO sail_data_source::formats::parquet::bucketed_sink] Wrote bucket 4: 156154 rows to Users/davidlopez/Proyec

  orders (bucketed by o_orderkey): 5549ms
  part: 707ms
  partsupp: 4116ms
  region: 2ms
  supplier: 105ms

  Running Q5...
  Q5 run 1: 317ms
  Q5 run 2: 282ms
  Q5 run 3: 268ms

Server stopped
Local results saved to /Users/davidlopez/Proyectos/LakeBench/examples/local_working_dir/sail_results.json


[2026-02-28T20:11:31Z INFO sail_python::spark::server] Shutting down the Spark Connect server...
[2026-02-28T20:11:31Z INFO sail_python::spark::server] The Spark Connect server has stopped.


## 6. Comparison

In [3]:
with open(RESULTS_FILE) as f:
    results = json.load(f)

pypi_write = results.get("pypi_write", [])
pypi_query = results.get("pypi_query", [])
local_write = results.get("local_write", [])
local_query = results.get("local_query", [])
pypi_ver = results.get("pypi_version", "?")
local_ver = results.get("local_version", "?")

print(f"PyPI Sail: {pypi_ver} (normal parquet)")
print(f"Local Sail: {local_ver} (bucketed parquet)")

for sf in SCALE_FACTORS:
    pypi_w = {r["table"]: r["ms"] for r in pypi_write if r["sf"] == sf}
    local_w = {r["table"]: r["ms"] for r in local_write if r["sf"] == sf}

    if not pypi_w and not local_w:
        print(f"\nSF={sf}: no data, skipping")
        continue

    hdr_pypi = f"PyPI ({pypi_ver})"
    hdr_local = f"Local ({local_ver})"

    print(f"\n{'='*80}")
    print(f"SF={sf} | Write times")
    print(f"{'='*80}")
    print(f"{'Table':<25} {hdr_pypi:>14} {hdr_local:>14} {'Diff':>10} {'Change':>10}")
    print(f"{'-'*25} {'-'*14} {'-'*14} {'-'*10} {'-'*10}")

    total_p, total_l = 0, 0
    for table_name, bucket_key in TPCH_TABLES:
        p = pypi_w.get(table_name, 0)
        l = local_w.get(table_name, 0)
        diff = l - p
        pct = (diff / p * 100) if p > 0 else 0
        sign = "+" if diff > 0 else ""
        tag = " *" if bucket_key else ""
        total_p += p
        total_l += l
        print(f"{table_name + tag:<25} {p:>12}ms {l:>12}ms {sign}{diff:>8}ms {sign}{pct:>8.1f}%")

    total_diff = total_l - total_p
    total_pct = (total_diff / total_p * 100) if total_p > 0 else 0
    sign = "+" if total_diff > 0 else ""
    print(f"{'-'*25} {'-'*14} {'-'*14} {'-'*10} {'-'*10}")
    print(f"{'TOTAL':<25} {total_p:>12}ms {total_l:>12}ms {sign}{total_diff:>8}ms {sign}{total_pct:>8.1f}%")
    print(f"\n  * = bucketed in local build")

    # Query comparison
    pypi_q = [r["ms"] for r in pypi_query if r["sf"] == sf]
    local_q = [r["ms"] for r in local_query if r["sf"] == sf]

    print(f"\n{'='*80}")
    print(f"SF={sf} | TPC-H Q5 (3 runs, lower is better)")
    print(f"{'='*80}")
    print(f"{'Run':<10} {hdr_pypi:>14} {hdr_local:>14} {'Diff':>10} {'Change':>10}")
    print(f"{'-'*10} {'-'*14} {'-'*14} {'-'*10} {'-'*10}")

    runs = min(len(pypi_q), len(local_q))
    if runs == 0:
        print(f"  (no query data for this SF)")
    else:
        for i in range(runs):
            p, l = pypi_q[i], local_q[i]
            diff = l - p
            pct = (diff / p * 100) if p > 0 else 0
            sign = "+" if diff > 0 else ""
            print(f"{'Run ' + str(i+1):<10} {p:>12}ms {l:>12}ms {sign}{diff:>8}ms {sign}{pct:>8.1f}%")

        avg_p = sum(pypi_q[:runs]) // runs
        avg_l = sum(local_q[:runs]) // runs
        avg_diff = avg_l - avg_p
        avg_pct = (avg_diff / avg_p * 100) if avg_p > 0 else 0
        sign = "+" if avg_diff > 0 else ""
        print(f"{'-'*10} {'-'*14} {'-'*14} {'-'*10} {'-'*10}")
        print(f"{'AVG':<10} {avg_p:>12}ms {avg_l:>12}ms {sign}{avg_diff:>8}ms {sign}{avg_pct:>8.1f}%")

PyPI Sail: 0.5.1 (normal parquet)
Local Sail: 0.5.1 (bucketed parquet)

SF=10 | Write times
Table                       PyPI (0.5.1)  Local (0.5.1)       Diff     Change
------------------------- -------------- -------------- ---------- ----------
customer                           434ms          344ms      -90ms    -20.7%
lineitem *                        4682ms         3068ms    -1614ms    -34.5%
nation                               2ms            2ms        0ms      0.0%
orders *                          1091ms         2712ms +    1621ms +   148.6%
part                               263ms          242ms      -21ms     -8.0%
partsupp                           757ms          722ms      -35ms     -4.6%
region                               2ms            2ms        0ms      0.0%
supplier                            36ms           33ms       -3ms     -8.3%
------------------------- -------------- -------------- ---------- ----------
TOTAL                             7267ms         7125ms 

## 7. Restore PyPI Sail

In [4]:
import subprocess
subprocess.run(["uv", "pip", "install", "pysail"], check=True)
print("PyPI Sail restored.")

PyPI Sail restored.


Using Python 3.13.11 environment at: /Users/davidlopez/Proyectos/LakeBench/.venv
Audited 1 package in 15ms
